# Unidad 4: DataFrames y Spark SQL

**Tecnicatura en Datos – Procesamiento con Apache Spark**  
Unidad 4 de 10 — Duración estimada: 2:30 hs

> **Nota Databricks:** `spark` ya está disponible. Los archivos CSV/JSON referencian rutas DBFS (`/mnt/...`). Los ejemplos con `createDataFrame` funcionan directamente.

In [0]:
from pyspark.sql.functions import col, when, upper, coalesce, lit
from pyspark.sql.functions import round as spark_round
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, DateType,
    DecimalType, ArrayType
)
print("Imports OK | Spark", spark.version)

---
## 1. RDD vs DataFrame

In [0]:
# Mismo filtro: RDD vs DataFrame
data = [(1, "Ana", 1500.0), (2, "Luis", 800.0), (3, "Marta", 2200.0)]

# Con RDD (bajo nivel)
# rdd = spark.sparkContext.parallelize(data)
# rdd_filtrado = rdd.filter(lambda x: x[2] > 1000)
# print("RDD:", rdd_filtrado.collect())

# Con DataFrame (alto nivel)
df = spark.createDataFrame(data, ["id", "cliente", "monto"])
print("\nDataFrame:")
df.display()
df_filtrado = df.filter(df.monto > 1000)
print("df filtrado:")
df_filtrado.display()

---
## 2. Schema

### Ejemplo 1 — Simple: inferencia automática desde CSV

> En Databricks: reemplazar la ruta por `/mnt/tu-datalake/ventas.csv`

In [0]:
# Simulamos la lectura usando createDataFrame
# En producción: df = spark.read.csv("/mnt/datalake/ventas.csv", header=True, inferSchema=True)

data_ventas = [
    (1, "manzana", 150.5, "2024-01-10"),
    (2, "banana",   80.0, "2024-01-11"),
    (3, "naranja", 200.0, "2024-01-12"),
]
df_inferido = spark.createDataFrame(data_ventas, ["id", "producto", "monto", "fecha"])

print("=== Schema inferido ===")
df_inferido.printSchema()
df_inferido.display()
# OPCIONAL display(df_inferido)

**Resultado esperado:**
```
root
 |-- id: long (nullable = true)
 |-- producto: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- fecha: string (nullable = true)   # ⚠ string en lugar de date
```

### Ejemplo 2 — Medio: schema explícito con tipos correctos

In [0]:
from pyspark.sql import Row
import datetime

schema = StructType([
    StructField("id",       IntegerType(), nullable=False),
    StructField("producto", StringType(),  nullable=True),
    StructField("monto",    DoubleType(),  nullable=True),
    StructField("fecha",    DateType(),    nullable=True),
])

data_tipada = [
    (1, "manzana", 150.5, datetime.date(2024, 1, 10)),
    (2, "banana",   80.0, datetime.date(2024, 1, 11)),
    (3, "naranja", 200.0, datetime.date(2024, 1, 12)),
]
df_explicito = spark.createDataFrame(data_tipada, schema)

print("=== Schema explícito ===")
df_explicito.printSchema()
df_explicito.display()

**Resultado esperado:**
```
root
 |-- id: integer (nullable = false)
 |-- producto: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- fecha: date (nullable = true)     # ✅ DateType correcto
```

### Ejemplo 3 — Avanzado: schema con tipos complejos (JSON anidado)

In [0]:
# Schema para JSON con array de structs
schema_pedido = StructType([
    StructField("pedido_id", IntegerType(), False),
    StructField("cliente",   StringType(),  True),
    StructField("items", ArrayType(
        StructType([
            StructField("producto", StringType(), True),
            StructField("cantidad", IntegerType(), True),
            StructField("precio",   DoubleType(),  True),
        ])
    ), True),
])

# Leer desde JSON en Databricks:
# df = spark.read.schema(schema_pedido).json("/mnt/datalake/pedidos/")

# Simulación en memoria (compatible con serverless):
import json
import pandas as pd

data_json = [
    '{"pedido_id": 1, "cliente": "Laura", "items": [{"producto": "laptop", "cantidad": 1, "precio": 1200.0}, {"producto": "mouse", "cantidad": 2, "precio": 25.0}]}',
    '{"pedido_id": 2, "cliente": "Carlos", "items": [{"producto": "teclado", "cantidad": 1, "precio": 80.0}]}',
]

# Parsear JSON y crear pandas DataFrame primero
parsed_data = [json.loads(j) for j in data_json]
pd_df = pd.DataFrame(parsed_data)

# Convertir a Spark DataFrame con schema explícito
df_pedidos = spark.createDataFrame(pd_df, schema=schema_pedido)

df_pedidos.printSchema()
df_pedidos.select("pedido_id", "cliente", "items").display()

In [0]:
data_json = [
    '{"pedido_id": 1, "cliente": "Laura", "items": [{"producto": "laptop", "cantidad": 1, "precio": 1200.0}, {"producto": "mouse", "cantidad": 2, "precio": 25.0}]}',
    '{"pedido_id": 2, "cliente": "Carlos", "items": [{"producto": "teclado", "cantidad": 1, "precio": 80.0}]}',
]


parsed_data = [json.loads(j) for j in data_json]
pd_df = pd.DataFrame(parsed_data)
df_inferido = spark.createDataFrame(parsed_data)
df_inferido.printSchema()
df_inferido.display()


**Resultado esperado:**
```
root
 |-- pedido_id: integer (nullable = false)
 |-- cliente: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct
 |    |    |-- producto: string
 |    |    |-- cantidad: integer
 |    |    |-- precio: double
```

---
## 3. Operaciones básicas con DataFrames

### Ejemplo 1 — Simple: `filter` y `select`

In [0]:
from pyspark.sql.functions import sum

data = [
    (1, "Ana",   1500.0, "AR"),
    (2, "Luis",   800.0, "MX"),
    (3, "Marta", 2200.0, "AR"),
    (5, "Maria", 2500.0, "CO"),
    (6, "Jose", 2200.0, "ES"),
    (4, "Pedro",  450.0, "CL"),
]
df = spark.createDataFrame(data, ["id", "cliente", "monto", "pais"])

df.display()
#selecT pais, monto, SUM(monto) as total from df where monto > 1000 group by pais
resultado = (
    df
    .filter(col("monto") > 1000)
    .select("pais", "monto")
    .groupBy("pais")
    .agg(sum("monto").alias("total"))
)

resultado.display()

**Resultado esperado:**
```
+-------+------+----+
|cliente| monto|pais|
+-------+------+----+
|    Ana|1500.0|  AR|
|  Marta|2200.0|  AR|
+-------+------+----+
```

### Ejemplo 2 — Medio: `withColumn`, `withColumnRenamed`, `drop`

In [0]:
TIPO_CAMBIO = 1000.0
"""
Agregar col -> withColumn
Renombrar col -> withColumnRenamed
Eliminar col -> drop
"""
df_transformado = (
    df
    .withColumn("monto_usd", spark_round(col("monto") / TIPO_CAMBIO, 2))
    .withColumnRenamed("cliente", "nombre_cliente")
    .drop("id")
)
df_transformado.display()

**Resultado esperado:**
```
+--------------+------+----+---------+
|nombre_cliente| monto|pais|monto_usd|
+--------------+------+----+---------+
|           Ana|1500.0|  AR|      1.5|
|          Luis| 800.0|  MX|      0.8|
|         Marta|2200.0|  AR|      2.2|
|         Pedro| 450.0|  CL|     0.45|
+--------------+------+----+---------+
```

### Ejemplo 3 — Avanzado: pipeline completo de limpieza

In [0]:
data_cruda = [
    (1, "ana",   1500.0, "AR"),
    (2, "luis",   None,  "MX"),
    (3, "marta", 2200.0,  None),
    (4, "pedro",  450.0, "CL"),
    (5, "sofia", 5800.0, "AR"),
]
df_crudo = spark.createDataFrame(data_cruda, ["id", "cliente", "monto", "pais"])

df_limpio = (
    df_crudo
    .fillna({"monto": 0.0, "pais": "DESCONOCIDO"})
    .withColumn("cliente", upper(col("cliente")))
    .withColumn("categoria",
        when(col("monto") < 1000, "bajo")
        .when(col("monto") < 3000, "medio")
        .otherwise("alto")
    )
    .orderBy(col("monto").desc())
    .select("id", "cliente", "pais", "monto", "categoria")
)
df_limpio.display()

**Resultado esperado:**
```
+---+-------+-----------+------+---------+
| id|cliente|       pais| monto|categoria|
+---+-------+-----------+------+---------+
|  5|  SOFIA|         AR|5800.0|     alto|
|  3|  MARTA|DESCONOCIDO|2200.0|    medio|
|  1|    ANA|         AR|1500.0|    medio|
|  4|  PEDRO|         CL| 450.0|     bajo|
|  2|   LUIS|         MX|   0.0|     bajo|
+---+-------+-----------+------+---------+
```

In [0]:
df = spark.read.csv('/Volumes/workspace/default/archivos_entrada/ventas_crudas.csv', header=True, inferSchema=True)
df.display()
df_limpio = (
    df
    .fillna({"monto": 0.0})
    .withColumn("cliente", upper(col("cliente")))
    .withColumn("categoria",
        when(col("monto") < 1000, "bajo")
        .when(col("monto") < 3000, "medio")
        .otherwise("alto")
    )
    .orderBy(col("monto").desc())
    .select("id_venta", "cliente",  "monto", "categoria")
)
df_limpio.display()

---
## 4. Spark SQL

### Ejemplo 1 — Simple: vista temporal y GROUP BY

In [0]:
data = [
    ("Ana",   1500.0, "AR"),
    ("Luis",   800.0, "MX"),
    ("Marta", 2200.0, "AR"),
    ("Pedro",  450.0, "CL"),
    ("Sofía", 5800.0, "AR"),
]
df_ventas = spark.createDataFrame(data, ["cliente", "monto", "pais"])
df_ventas.createOrReplaceTempView("ventas")

resultado = spark.sql("""
    SELECT pais, SUM(monto) AS total_vendido
    FROM ventas
    GROUP BY pais
    ORDER BY total_vendido DESC
""")
resultado.show()

**Resultado esperado:**
```
+----+-------------+
|pais|total_vendido|
+----+-------------+
|  AR|       9500.0|
|  MX|        800.0|
|  CL|        450.0|
+----+-------------+
```

### Ejemplo 2 — Medio: subconsulta con AVG

In [0]:
# Clientes con monto superior al promedio general
resultado = spark.sql("""
    SELECT cliente, monto, pais
    FROM ventas
    WHERE monto > (SELECT AVG(monto) FROM ventas)
    ORDER BY monto DESC
""")
resultado.show()

**Resultado esperado:**
```
+-------+------+----+
|cliente| monto|pais|
+-------+------+----+
|  Sofía|5800.0|  AR|
|  Marta|2200.0|  AR|
|    Ana|1500.0|  AR|
+-------+------+----+
```

> El promedio es (1500+800+2200+450+5800)/5 = 2150. Solo los tres primeros lo superan.

### Ejemplo 3 — Avanzado: CTE + window function + JOIN entre vistas

In [0]:
data_paises = [("AR", "Argentina"), ("MX", "México"), ("CL", "Chile")]
df_paises = spark.createDataFrame(data_paises, ["codigo", "nombre_pais"])
df_paises.createOrReplaceTempView("paises")
# La vista "ventas" sigue disponible de la celda anterior

resultado = spark.sql("""
    WITH ventas_enriquecidas AS (
        SELECT
            v.cliente,
            v.monto,
            v.pais,
            p.nombre_pais,
            RANK() OVER (PARTITION BY v.pais ORDER BY v.monto DESC) AS ranking_en_pais
        FROM ventas v
        JOIN paises p ON v.pais = p.codigo
    )
    SELECT *
    FROM ventas_enriquecidas
    WHERE ranking_en_pais = 1
    ORDER BY monto DESC
""")
resultado.show()

**Resultado esperado:**
```
+-------+------+----+-----------+---------------+
|cliente| monto|pais|nombre_pais|ranking_en_pais|
+-------+------+----+-----------+---------------+
|  Sofía|5800.0|  AR|  Argentina|              1|
|   Luis| 800.0|  MX|     México|              1|
|  Pedro| 450.0|  CL|      Chile|              1|
+-------+------+----+-----------+---------------+
```

---
## 5. Práctica guiada

### Ejercicio 1 — Simple: JSON anidado

In [0]:
data_json = [
    '{"id": 1, "nombre": "Ana",   "direccion": {"ciudad": "Buenos Aires",    "pais": "AR"}}',
    '{"id": 2, "nombre": "Luis",  "direccion": {"ciudad": "Ciudad de México", "pais": "MX"}}',
    '{"id": 3, "nombre": "Marta", "direccion": {"ciudad": "Santiago",         "pais": "CL"}}',
]

rdd_json = spark.sparkContext.parallelize(data_json)
df = spark.read.json(rdd_json)

df.printSchema()
df.select("id", "nombre", "direccion.ciudad", "direccion.pais").show()

**Resultado esperado:**
```
+---+------+----------------+----+
| id|nombre|          ciudad|pais|
+---+------+----------------+----+
|  1|   Ana|    Buenos Aires|  AR|
|  2|  Luis|Ciudad de México|  MX|
|  3| Marta|        Santiago|  CL|
+---+------+----------------+----+
```

### Ejercicio 2 — Medio: comparar inferencia vs schema explícito

In [0]:
import time, datetime

# DataFrame con inferencia de tipos (string para fecha)
data_str = [(1, "manzana", 150.5, "2024-01-10"), (2, "banana", 80.0, "2024-01-11")]
t0 = time.time()
df_inf = spark.createDataFrame(data_str, ["id", "producto", "monto", "fecha"])
_ = df_inf.count()
t1 = time.time()
print(f"Sin schema explícito: {t1-t0:.4f}s")
df_inf.printSchema()

# DataFrame con schema explícito y DateType
schema = StructType([
    StructField("id",       IntegerType(), False),
    StructField("producto", StringType(),  True),
    StructField("monto",    DoubleType(),  True),
    StructField("fecha",    DateType(),    True),
])
data_date = [(1, "manzana", 150.5, datetime.date(2024,1,10)), (2, "banana", 80.0, datetime.date(2024,1,11))]
t2 = time.time()
df_exp = spark.createDataFrame(data_date, schema)
_ = df_exp.count()
t3 = time.time()
print(f"Con schema explícito: {t3-t2:.4f}s")
df_exp.printSchema()

---
## 6. Preguntas de revisión

1. ¿Por qué los DataFrames son más rápidos que los RDDs?
2. ¿Cuál es la desventaja de `inferSchema=True`?
3. ¿Qué tipo complejo usarías para representar una lista de objetos JSON?
4. ¿Qué significa que Spark SQL y la API de DataFrames usen el mismo motor?
5. ¿Para qué sirve `createOrReplaceTempView`?

---
**Próxima unidad:** Transformaciones avanzadas